# 04 · Consultas DuckDB

Objetivo: comprobar tamaños, comparar coberturas y consultar controles de calidad. Cada SELECT opera sobre una base privada generada en esta sesión. La carpeta `sql/` permite revisar las consultas históricas completas.

**Requisitos.** Python, ZIP y dependencias fijadas. [SQL histórico](../sql/) · [Diccionario](../docs/diccionario_datos.md).

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import subprocess
import sys
import tempfile
import zipfile

package_name = "SLP-BigData-Reproducible"
here = Path.cwd().resolve()
explicit = os.getenv("SLP_BIGDATA_ROOT")
candidates = [Path(explicit).expanduser().resolve()] if explicit else [
    here, here.parent, here / package_name,
]
ROOT = next((p for p in candidates if (p / "src" / "repro.py").is_file()), None)
colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if ROOT is None and colab:
    from google.colab import files
    print("Selecciona el ZIP SLP-BigData-Reproducible.zip.")
    uploaded = files.upload()
    if set(uploaded) != {package_name + ".zip"}:
        raise ValueError("Selecciona únicamente el ZIP de distribución del proyecto.")
    with zipfile.ZipFile(package_name + ".zip") as archive:
        for entry in archive.infolist():
            parts = Path(entry.filename).parts
            if not parts or parts[0] != package_name or ".." in parts or "\\" in entry.filename:
                raise ValueError("El ZIP contiene una ruta ajena al paquete.")
        archive.extractall(here)
    ROOT = here / package_name
if ROOT is None or not (ROOT / "src" / "repro.py").is_file():
    raise FileNotFoundError("Abre la carpeta descomprimida o define SLP_BIGDATA_ROOT.")

requirements = ROOT / "requirements.txt"
def current_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None
pins = [line.split("==", 1) for line in requirements.read_text(encoding="utf-8").splitlines()
        if "==" in line and not line.startswith("#")]
missing = [(name, version) for name, version in pins if current_version(name) != version]
if missing:
    if not colab:
        raise RuntimeError("Instala requirements.txt en el entorno virtual activo antes de continuar.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
sys.path.insert(0, str(ROOT))
from src.repro import run_demo, run_original, run_benchmark, query, read_report

private = ROOT / ".bigdata-private"
private.mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix="notebook-", dir=private))
print("Dependencias verificadas:", {name: current_version(name) for name, _ in pins})

In [ ]:
# A ejecuta una demostración sintética. B verifica archivos autorizados.
MODO = "A"
INAH = private / "input-inah"
DATATUR = private / "input-datatur"

def ejecutar(modo, salida):
    if modo == "A":
        return run_demo(salida)
    if modo == "B":
        if not INAH.is_dir() or not DATATUR.is_dir():
            print("Modo B requiere los originales autorizados. Consulta docs/reproducibilidad.md.")
            return None
        return run_original(INAH, DATATUR, salida)
    raise ValueError("MODO admite A o B.")

In [ ]:
producto = ejecutar(MODO, WORK / "consultas")
if producto is not None:
    db = producto / "warehouse" / "campaign.duckdb"
    consultas = {
        "Cobertura INAH": '''
            SELECT site_id, min(period) AS inicio, max(period) AS final,
                   count(*) AS meses
            FROM fact_inah_visits_monthly GROUP BY site_id ORDER BY site_id
        ''',
        "Cobertura DataTur": '''
            SELECT center_id, min(period) AS inicio, max(period) AS final,
                   count(*) AS meses, count(hotel_occupancy_percent) AS tasas
            FROM fact_datatur_occupancy_monthly GROUP BY center_id ORDER BY center_id
        ''',
        "Duplicados INAH": '''
            SELECT site_id, period, count(*) AS repeticiones
            FROM fact_inah_visits_monthly GROUP BY site_id, period HAVING count(*) > 1
        ''',
    }
    for titulo, sentencia in consultas.items():
        print(titulo, query(db, sentencia))

**Interpretación.** El conteo de tasas observadas puede ser menor que el de periodos. Las series se exploran por su propio grano. El listado de claves duplicadas debe quedar vacío. [Resultados históricos](../docs/resultados_verificados.md).